In [11]:
import os
import json
import re
import pandas as pd
from datasets import load_dataset, load_from_disk
from openai import OpenAI
from tqdm import tqdm

In [ ]:
os.environ["OPENAI_API_KEY"] = "YOUR_PERSONAL_KEY"

In [3]:
EVALUATION_PROMPT = """You are an expert linguist specializing in translation quality evaluation.

IMPORTANT: Evaluate each translation independently. Do not compare with previous examples.

Task:
Evaluate the given translation from {source_lang} to {target_lang} on a scale of 0 to 10.
Consider the **domain**: {domain} (e.g., banking, everyday conversation, sports, legal, medical, etc.).
The translation must be appropriate for this domain in terms of terminology and style.

Input:
- Source text ({source_lang}): "{source_text}"
- Translation ({target_lang}): "{translation}"

Scoring criteria (total 0–10):
1. **Accuracy & completeness** (0–4 points)  
   - All key information is preserved.  
   - No additions, omissions, or distortions.
   - **For idioms and cultural references**: prefer adaptive/idiomatic translation over literal word-for-word.
2. **Grammar & orthography** (0–3 points)  
   - Correct grammar, spelling, punctuation.  
   - No errors that hinder understanding.
3. **Naturalness & style** (0–3 points)  
   - Fluent, idiomatic, appropriate for the domain.  
   - Reads as if originally written in {target_lang}.

Instructions:
1. Assign points for each criterion, sum them, and provide the total score (0–10).
2. Write a brief comment (1–2 sentences) explaining why the points were deducted.
3. Return ONLY valid JSON in this format:

{{
  "score": <integer>,
  "comment": "<string>"
}}
"""

In [4]:
# Убедимся, что llm отвечает

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), base_url="http://31.56.222.86:8002/api/providers/openai/v1")
try:
    resp = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": "Say hello"}],
        max_tokens=10
    )
    print("API works:", resp.choices[0].message.content)
except Exception as e:
    print("API error:", e)

API works: Hello! How can I help you today?


In [32]:
def evaluate_dataset(dataset_type="classification",
                     hf_original_name=None,
                     hf_translation_name=None,
                     label_column="label",
                     hf_corpus_original=None,
                     hf_corpus_translation=None,
                     hf_queries_original=None,
                     hf_queries_translation=None,
                     corpus_original_config=None,
                     corpus_translation_config=None,
                     queries_original_config=None,
                     queries_translation_config=None,
                     local_path=None,
                     api_key=None,
                     src_lang="English",
                     tgt_lang="Russian",
                     domain="general",
                     n_per_class=10,
                     n_corpus=40,
                     n_queries=30,
                     n_history=30):
    client = OpenAI(api_key=api_key, base_url="http://31.56.222.86:8002/api/providers/openai/v1")
    results = []

    if dataset_type == "classification":
        if hf_original_name is None or hf_translation_name is None:
            raise ValueError("Укажите hf_original_name и hf_translation_name")
        
        print(f"Загрузка {hf_original_name}")
        orig_ds = load_dataset(hf_original_name, trust_remote_code=True)["train"].to_pandas()
        print(f"Загрузка {hf_translation_name}")
        trans_ds = load_dataset(hf_translation_name, trust_remote_code=True)["train"].to_pandas()
        
        print("Колонки оригинала:", list(orig_ds.columns))
        print("Колонки перевода:", list(trans_ds.columns))
        
        # Проверяем наличие id для объединения
        if "id" in orig_ds.columns and "id" in trans_ds.columns:
            print("Объединение по id")
            merged = pd.merge(orig_ds, trans_ds, on="id", how="inner")
            # Переименовываем колонки текста
            if "text_x" in merged.columns and "text_y" in merged.columns:
                merged = merged.rename(columns={"text_x": "text", "text_y": "text_ru"})
            # Если есть и другие суффиксы, можно их тоже обработать, но оставим так
        else:
            # Нет id — предполагаем, что строки идут в одинаковом порядке и количество строк совпадает
            print("Нет id. Объединяем по индексу (порядку строк). Убедитесь, что порядок совпадает!")
            if len(orig_ds) != len(trans_ds):
                print(f"Предупреждение: разное количество строк: {len(orig_ds)} vs {len(trans_ds)}. Обрезаем до минимума.")
                min_len = min(len(orig_ds), len(trans_ds))
                orig_ds = orig_ds.iloc[:min_len].reset_index(drop=True)
                trans_ds = trans_ds.iloc[:min_len].reset_index(drop=True)
            else:
                orig_ds = orig_ds.reset_index(drop=True)
                trans_ds = trans_ds.reset_index(drop=True)
            
            # Создаём merged копируя orig_ds и добавляя text_ru из trans_ds
            merged = orig_ds.copy()
            # Если в trans_ds колонка называется 'text', переименуем в 'text_ru'
            if "text" in trans_ds.columns:
                merged["text_ru"] = trans_ds["text"].values
            else:
                # Ищем любую колонку, похожую на текст
                text_candidates = [col for col in trans_ds.columns if "text" in col.lower()]
                if text_candidates:
                    merged["text_ru"] = trans_ds[text_candidates[0]].values
                else:
                    raise ValueError("Не найдена колонка с переводом в trans_ds")
            
            # Если в orig_ds колонка текста называется не 'text', переименуем
            if "text" not in merged.columns:
                text_candidates = [col for col in merged.columns if "text" in col.lower()]
                if text_candidates:
                    merged = merged.rename(columns={text_candidates[0]: "text"})
                else:
                    raise ValueError("Не найдена колонка с оригиналом в orig_ds")
        
        # Определяем колонку с меткой
        if label_column not in merged.columns:
            if f"{label_column}_ru" in merged.columns:
                label_column = f"{label_column}_ru"
            elif label_column in orig_ds.columns:
                merged[label_column] = orig_ds[label_column].values
            else:
                # Ищем возможные колонки с метками
                possible_labels = [col for col in merged.columns if "label" in col.lower()]
                if possible_labels:
                    label_column = possible_labels[0]
                    print(f"Используем колонку меток: {label_column}")
                else:
                    raise ValueError(f"Колонка меток '{label_column}' не найдена")
        
        # Удаляем строки с пустыми значениями
        merged = merged.dropna(subset=["text", "text_ru", label_column])
        print(f"Всего пар после очистки: {len(merged)}")
        print("Распределение по классам:\n", merged[label_column].value_counts())
        
        # Стратифицированная выборка
        sampled_dfs = []
        for lbl, group in merged.groupby(label_column):
            n_samples = min(n_per_class, len(group))
            sampled_dfs.append(group.sample(n_samples, random_state=42))
        sampled_df = pd.concat(sampled_dfs)
        print(f"Выбрано {len(sampled_df)} примеров (по {n_per_class} на класс)")
        
        # Оценка
        for _, row in tqdm(sampled_df.iterrows(), total=len(sampled_df), desc="Оценка"):
            results.append(get_score(
                client, row["text"], row["text_ru"], src_lang, tgt_lang, domain,
                source="classification", type_=f"class_{row[label_column]}"
            ))
    
    # RETRIEVAL (HF или локальный)
    elif dataset_type == "retrieval":
        # Вариант 1: загрузка с Hugging Face (с указанием конфигураций)
        if (hf_corpus_original and hf_corpus_translation and 
            hf_queries_original and hf_queries_translation):
            
            # Корпус
            print("Загрузка корпуса с Hugging Face...")
            corpus_orig = load_dataset(
                hf_corpus_original, 
                corpus_original_config, 
                trust_remote_code=True
            )["train"].to_pandas()
            corpus_trans = load_dataset(
                hf_corpus_translation,
                corpus_translation_config,
                trust_remote_code=True
            )["train"].to_pandas()
            
            if "id" in corpus_orig.columns and "id" in corpus_trans.columns:
                corpus = pd.merge(corpus_orig, corpus_trans, on="id", how="inner")
                if "text_x" in corpus.columns and "text_y" in corpus.columns:
                    corpus = corpus.rename(columns={"text_x": "text", "text_y": "text_ru"})
            else:
                print("Корпус: нет id, объединяем по индексу")
                corpus_orig = corpus_orig.reset_index(drop=True)
                corpus_trans = corpus_trans.reset_index(drop=True)
                corpus = pd.concat([corpus_orig, corpus_trans], axis=1)
                if "text" in corpus_orig and "text" in corpus_trans:
                    corpus = corpus.rename(columns={"text_x": "text", "text_y": "text_ru"})
            corpus = corpus.dropna(subset=["text", "text_ru"])
            
            # Запросы
            print("Загрузка запросов с Hugging Face...")
            queries_orig = load_dataset(
                hf_queries_original,
                queries_original_config,
                trust_remote_code=True
            )["train"].to_pandas()
            queries_trans = load_dataset(
                hf_queries_translation,
                queries_translation_config,
                trust_remote_code=True
            )["train"].to_pandas()
            
            if "id" in queries_orig.columns and "id" in queries_trans.columns:
                queries = pd.merge(queries_orig, queries_trans, on="id", how="inner")
                if "text_x" in queries.columns and "text_y" in queries.columns:
                    queries = queries.rename(columns={"text_x": "text", "text_y": "text_ru"})
            else:
                print("Запросы: нет id, объединяем по индексу")
                queries_orig = queries_orig.reset_index(drop=True)
                queries_trans = queries_trans.reset_index(drop=True)
                queries = pd.concat([queries_orig, queries_trans], axis=1)
                if "text" in queries_orig and "text" in queries_trans:
                    queries = queries.rename(columns={"text_x": "text", "text_y": "text_ru"})
            queries = queries.dropna(subset=["text", "text_ru"])
            
            # История (если есть)
            if "history" in queries_orig.columns and "history_ru" in queries_trans.columns:
                if "id" in queries_orig and "id" in queries_trans:
                    queries_hist = pd.merge(
                        queries_orig[["id", "history"]],
                        queries_trans[["id", "history_ru"]],
                        on="id", how="inner"
                    )
                else:
                    print("История: объединяем по индексу")
                    hist_orig = queries_orig[["history"]].reset_index(drop=True)
                    hist_trans = queries_trans[["history_ru"]].reset_index(drop=True)
                    queries_hist = pd.concat([hist_orig, hist_trans], axis=1).dropna()
            else:
                queries_hist = pd.DataFrame()
                print("Поля history/history_ru не найдены в запросах.")
        
        # Вариант 2: локальный режим (папки corpus и queries)
        elif local_path is not None:
            print(f"Загрузка из локального пути: {local_path}")
            corpus_path = os.path.join(local_path, "corpus")
            queries_path = os.path.join(local_path, "queries")
            
            # Корпус
            corpus = load_from_disk(corpus_path)["train"].to_pandas()
            if not {"text", "text_ru"}.issubset(corpus.columns):
                raise ValueError(f"В {corpus_path} отсутствуют колонки 'text' или 'text_ru'")
            corpus = corpus.dropna(subset=["text", "text_ru"])
            
            # Запросы
            queries = load_from_disk(queries_path)["train"].to_pandas()
            if not {"text", "text_ru"}.issubset(queries.columns):
                raise ValueError(f"В {queries_path} отсутствуют колонки 'text' или 'text_ru'")
            queries = queries.dropna(subset=["text", "text_ru"])
            
            # История (если есть)
            if {"history", "history_ru"}.issubset(queries.columns):
                queries_hist = queries[["history", "history_ru"]].dropna()
            else:
                queries_hist = pd.DataFrame()
                print("Поля history/history_ru не найдены в запросах, история пропущена.")
        
        else:
            raise ValueError("Для retrieval укажите либо HF-датасеты (с конфигурациями), либо local_path")
        
        # Оценка retrieval
        # Корпус
        for _, row in tqdm(corpus.sample(min(n_corpus, len(corpus)), random_state=42).iterrows(),
                           total=min(n_corpus, len(corpus)), desc="Corpus texts"):
            results.append(get_score(client, row["text"], row["text_ru"], src_lang, tgt_lang, domain,
                                    "corpus", "text"))
        
        # Вопросы
        for _, row in tqdm(queries.sample(min(n_queries, len(queries)), random_state=42).iterrows(),
                           total=min(n_queries, len(queries)), desc="Queries questions"):
            results.append(get_score(client, row["text"], row["text_ru"], src_lang, tgt_lang, domain,
                                    "queries", "question"))
        
        # История
        if not queries_hist.empty:
            for _, row in tqdm(queries_hist.sample(min(n_history, len(queries_hist)), random_state=42).iterrows(),
                               total=min(n_history, len(queries_hist)), desc="Queries history"):
                results.append(get_score(client, row["history"], row["history_ru"], src_lang, tgt_lang, domain,
                                        "queries", "history"))
    
    else:
        raise ValueError("dataset_type должен быть 'classification' или 'retrieval'")
    
    return pd.DataFrame(results)

In [29]:
# Выставляем оценку переводу 

def get_score(client, src, tgt, src_lang, tgt_lang, domain, source, type_):
    try:
        resp = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a professional translation evaluator. Return ONLY valid JSON. No explanations, no markdown, no extra text."},
                {"role": "user", "content": EVALUATION_PROMPT.format(
                    source_lang=src_lang, target_lang=tgt_lang, domain=domain,
                    source_text=src, translation=tgt
                )}
            ],
            temperature=0,
            max_tokens=200
        )
        
        content = resp.choices[0].message.content.strip()
        
        # Пытаемся извлечь JSON любым способом
        # Способ 1: ищем { ... } с учётом переносов
        json_match = re.search(r'\{[^{}]*\}', content, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
        else:
            # Способ 2: ищем score и comment через regex
            score_match = re.search(r'score["\s:]+(\d+)', content, re.IGNORECASE)
            comment_match = re.search(r'comment["\s:]+"([^"]+)"', content, re.IGNORECASE)
            
            if score_match:
                return {
                    "source": source, "type": type_,
                    "src_lang": src_lang, "tgt_lang": tgt_lang, "domain": domain,
                    "original": src, "translation": tgt,
                    "score": int(score_match.group(1)),
                    "comment": comment_match.group(1) if comment_match else ""
                }
            else:
                # Способ 3: просто ищем любое число
                numbers = re.findall(r'\d+', content)
                if numbers:
                    return {
                        "source": source, "type": type_,
                        "src_lang": src_lang, "tgt_lang": tgt_lang, "domain": domain,
                        "original": src, "translation": tgt,
                        "score": min(10, int(numbers[0])),
                        "comment": content[:100]
                    }
                raise ValueError("No score found")
        
        # Парсим найденный JSON
        # Очищаем от возможных проблем
        json_str = json_str.replace('\n', ' ').replace('\r', ' ')
        data = json.loads(json_str)
        
        return {
            "source": source, "type": type_,
            "src_lang": src_lang, "tgt_lang": tgt_lang, "domain": domain,
            "original": src, "translation": tgt,
            "score": data.get("score", 5),
            "comment": data.get("comment", "")
        }
        
    except Exception as e:
        print(f"Error for {source}/{type_}: {e}")
        # Возвращаем заглушку вместо None, чтобы не ломать аналитику
        return {
            "source": source, "type": type_,
            "src_lang": src_lang, "tgt_lang": tgt_lang, "domain": domain,
            "original": src, "translation": tgt,
            "score": 5,
            "comment": f"Parse error: {str(e)[:50]}"
        }


In [33]:
# Вызов для classification

df_class = evaluate_dataset(
    dataset_type="classification",
    hf_original_name="DeepPavlov/atis_intent_classification",
    hf_translation_name="DeepPavlov/atis_intent_classification_ru",
    label_column="label",
    api_key=os.getenv("OPENAI_API_KEY"),
    domain="aviation",
    n_per_class=5
)

Загрузка DeepPavlov/atis_intent_classification
Загрузка DeepPavlov/atis_intent_classification_ru
Колонки оригинала: ['text', 'label', 'label_text']
Колонки перевода: ['text', 'label', 'label_text', 'label_text_ru']
Нет id. Объединяем по индексу (порядку строк). Убедитесь, что порядок совпадает!
Всего пар после очистки: 4834
Распределение по классам:
 label
0    3666
2     423
4     255
5     157
6     147
3      81
1      54
7      51
Name: count, dtype: int64
Выбрано 40 примеров (по 5 на класс)


Оценка: 100%|██████████| 40/40 [00:35<00:00,  1.12it/s]


In [36]:
# Статистика и средняя оценка

print("Общая статистика по всем переводам:")
print(f"Всего оценено примеров: {len(df_class)}")
print(f"Средняя оценка: {df_class['score'].mean():.2f} / 10")
print(f"Минимальная оценка: {df_class['score'].min()}")
print(f"Максимальная оценка: {df_class['score'].max()}")
print()

print("Статистика по классам")
# Группировка по классу
class_stats = df_class.groupby('type')['score'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
class_stats = class_stats.round(2)
print(class_stats.to_string())
print()

print("Распределение оценок по классам (процентили)")
percentiles = df_class.groupby('type')['score'].describe(percentiles=[.25, .5, .75]).round(2)
print(percentiles.to_string())

Общая статистика по всем переводам:
Всего оценено примеров: 40
Средняя оценка: 8.70 / 10
Минимальная оценка: 2
Максимальная оценка: 10

Статистика по классам
         count  mean  median   std  min  max
type                                        
class_0      5   9.0     9.0  0.71    8   10
class_1      5   9.2     9.0  0.45    9   10
class_2      5   9.0     9.0  1.00    8   10
class_3      5   9.2    10.0  1.30    7   10
class_4      5   9.4     9.0  0.55    9   10
class_5      5   8.2    10.0  3.49    2   10
class_6      5   7.6     9.0  3.21    2   10
class_7      5   8.0     8.0  1.58    6   10

Распределение оценок по классам (процентили)
         count  mean   std  min  25%   50%   75%   max
type                                                  
class_0    5.0   9.0  0.71  8.0  9.0   9.0   9.0  10.0
class_1    5.0   9.2  0.45  9.0  9.0   9.0   9.0  10.0
class_2    5.0   9.0  1.00  8.0  8.0   9.0  10.0  10.0
class_3    5.0   9.2  1.30  7.0  9.0  10.0  10.0  10.0
class_4    5.0  

In [35]:
# Вызов для retrieval

df_ret_hf = evaluate_dataset(
    dataset_type="retrieval",
    hf_corpus_original="DeepPavlov/canard",
    hf_corpus_translation="DeepPavlov/canard_ru",
    corpus_original_config="corpus",
    corpus_translation_config="corpus",
    hf_queries_original="DeepPavlov/canard",
    hf_queries_translation="DeepPavlov/canard_ru",
    queries_original_config="queries",
    queries_translation_config="queries",
    api_key=os.getenv("OPENAI_API_KEY"),
    n_corpus=40,
    n_queries=30,
    n_history=30
)

Загрузка корпуса с Hugging Face...
Загрузка запросов с Hugging Face...


Queries history: 100%|██████████| 30/30 [00:32<00:00,  1.07s/it]


In [37]:
# Статистика и средняя оценка

print("Общая статистика по всем переводам:")
print(f"Всего оценено примеров: {len(df_ret_hf)}")
print(f"Средняя оценка: {df_ret_hf['score'].mean():.2f} / 10")
print(f"Минимальная оценка: {df_ret_hf['score'].min()}")
print(f"Максимальная оценка: {df_ret_hf['score'].max()}")
print()

print("Статистика по классам")
# Группировка по классу
class_stats = df_ret_hf.groupby('type')['score'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
class_stats = class_stats.round(2)
print(class_stats.to_string())
print()

print("Распределение оценок по классам (процентили)")
percentiles = df_ret_hf.groupby('type')['score'].describe(percentiles=[.25, .5, .75]).round(2)
print(percentiles.to_string())

Общая статистика по всем переводам:
Всего оценено примеров: 100
Средняя оценка: 7.96 / 10
Минимальная оценка: 5
Максимальная оценка: 10

Статистика по классам
          count  mean  median   std  min  max
type                                         
history      30  8.73     9.0  0.87    6   10
question     30  7.97     8.0  1.10    6   10
text         40  7.38     8.0  1.08    5   10

Распределение оценок по классам (процентили)
          count  mean   std  min   25%  50%  75%   max
type                                                  
history    30.0  8.73  0.87  6.0  8.00  9.0  9.0  10.0
question   30.0  7.97  1.10  6.0  7.00  8.0  9.0  10.0
text       40.0  7.38  1.08  5.0  6.75  8.0  8.0  10.0
